# Huấn luyện RT-DETR cho nhận diện biển báo (Zalo AI 2020)
Notebook này được thiết kế để chạy độc lập trên **Google Colab**. Nó sẽ dùng API tải dữ liệu, tự động cắt ảnh, xử lý JSON sang format YOLO và huấn luyện mô hình Transformer RT-DETR siêu nhanh.

In [ ]:
# Tải bộ dữ liệu Zalo AI từ Kaggle về Google Colab bằng API (Yêu cầu phải upload file kaggle.json lên Colab trước)
!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d phhasian0710/za-traffic-2020
!unzip -q -n za-traffic-2020.zip -d /content/dataset

In [ ]:
import os
import json
import glob
import shutil
import random
from tqdm import tqdm

# Thiết lập các đường dẫn thư mục cho Colab
json_paths = glob.glob('/content/dataset/**/train_traffic_sign_dataset.json', recursive=True)
if not json_paths:
    raise FileNotFoundError("Không tìm thấy file JSON. Vui lòng kiểm tra lại bước tải dữ liệu!")

json_path = json_paths[0]
image_dir = os.path.dirname(json_path).replace('traffic_train', 'traffic_train/images')

if not os.path.exists(image_dir):
    img_dirs = glob.glob('/content/dataset/**/traffic_train/images', recursive=True)
    if img_dirs:
        image_dir = img_dirs[0]

# Nơi chứa dữ liệu sau khi đã tiền xử lý
dataset_dir = '/content/yolo_dataset'

for split in ['train', 'val']:
    os.makedirs(f'{dataset_dir}/{split}/images', exist_ok=True)
    os.makedirs(f'{dataset_dir}/{split}/labels', exist_ok=True)

print("Đang đọc dữ liệu JSON...")
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

images_info = {img['id']: img for img in data['images']}

img_to_anns = {}
for ann in data['annotations']:
    img_id = ann['image_id']
    if img_id not in img_to_anns:
        img_to_anns[img_id] = []
    img_to_anns[img_id].append(ann)

image_ids = list(images_info.keys())
random.seed(42)
random.shuffle(image_ids)
split_idx = int(len(image_ids) * 0.8)
train_ids = image_ids[:split_idx]
val_ids = image_ids[split_idx:]

print(f"Tổng số ảnh: {len(image_ids)}. Train: {len(train_ids)}, Val: {len(val_ids)}")

In [ ]:
def convert_coco_to_yolo(bbox, img_width, img_height):
    x_min, y_min, w, h = bbox
    x_center = (x_min + w / 2) / img_width
    y_center = (y_min + h / 2) / img_height
    w_norm = w / img_width
    h_norm = h / img_height
    return x_center, y_center, w_norm, h_norm

print("Đang xử lý và tạo file .txt cho RT-DETR...")

def process_split(ids, split_name):
    for img_id in tqdm(ids, desc=f"Processing {split_name}"):
        img_info = images_info[img_id]
        img_filename = img_info['file_name']
        img_width = img_info['width']
        img_height = img_info['height']
        
        src_img_path = os.path.join(image_dir, img_filename)
        dst_img_path = os.path.join(dataset_dir, split_name, 'images', img_filename)
        
        if os.path.exists(src_img_path):
            shutil.copy(src_img_path, dst_img_path)
            
            txt_filename = img_filename.rsplit('.', 1)[0] + '.txt'
            txt_path = os.path.join(dataset_dir, split_name, 'labels', txt_filename)
            
            with open(txt_path, 'w') as f_txt:
                if img_id in img_to_anns:
                    for ann in img_to_anns[img_id]:
                        class_id = int(ann['category_id']) - 1
                        x_c, y_c, w_n, h_n = convert_coco_to_yolo(ann['bbox'], img_width, img_height)
                        f_txt.write(f"{class_id} {x_c:.6f} {y_c:.6f} {w_n:.6f} {h_n:.6f}\n")

process_split(train_ids, 'train')
process_split(val_ids, 'val')

print("Hoàn tất quá trình chuẩn bị dữ liệu!")

In [ ]:
yaml_content = f"""
path: {dataset_dir}
train: train/images
val: val/images

names:
  0: No entry
  1: No parking / waiting
  2: No turning
  3: Max Speed
  4: Other prohibition signs
  5: Warning signs
  6: Mandatory signs
"""

with open('/content/dataset.yaml', 'w', encoding='utf-8') as f:
    f.write(yaml_content.strip())
    
print("Đã tạo xong file cấu hình dataset.yaml")

In [ ]:
# Cài đặt thư viện ultralytics
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
from ultralytics import RTDETR
from google.colab import drive
import os

# Yêu cầu quyền truy cập Google Drive
drive.mount('/content/drive')
save_dir = '/content/drive/MyDrive/DoAn_NhanDienBienBao'
os.makedirs(save_dir, exist_ok=True)

# Khởi tạo mô hình RT-DETR bản Large
model = RTDETR('rtdetr-l.pt')

# Bắt đầu huấn luyện với cấu hình CHỐNG TRÀN RAM (Gradient Accumulation)
results = model.train(
    data='/content/dataset.yaml',
    epochs=50,
    imgsz=1280,         # [TỪ E3] Bắt buộc giữ độ phân giải cao
    batch=2,            # [QUAN TRỌNG] Ép batch=2 để GPU Colab không chết (OOM)
    accumulate=4,       # Tích lũy 4 mini-batch -> Batch ảo = 2x4 = 8
    optimizer='AdamW',  # Lịch trình học thuật
    cos_lr=True,        # Hạ nhiệt độ Learning Rate
    project=save_dir, 
    name='rtdetr_highres',
    device=0,
)



In [ ]:
# [TỪ E3, M4.2] THUẬT TOÁN DÀNH CHO WEB APP
!pip install -q sahi
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

try:
    detection_model = AutoDetectionModel.from_pretrained(
        model_type='rtdetr',
        model_path='/content/drive/MyDrive/DoAn_NhanDienBienBao/rtdetr_highres/weights/best.pt',
        confidence_threshold=0.25,
        device="cuda:0"
    )
    print("Hệ thống SAHI đã sẵn sàng cho RT-DETR trên Web App.")
except Exception as e:
    print("Chưa có weights để chạy SAHI:", e)
